# Module 02: Pandas for Machine Learning
## Notebook 01: Series and DataFrame Fundamentals

Pandas is the primary data manipulation and exploratory analysis library in the Python data science stack. While NumPy provides the raw numeric compute engine, Pandas provides the labeled, multi-type tabular structures required for real-world machine learning datasets.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Construct and inspect 1D `pd.Series` and 2D `pd.DataFrame` structures.
2. Read and write tabular datasets from **CSV** and **Excel (`.xlsx`)** files with custom delimiters, column filtering (`usecols`), streaming chunks (`chunksize`), and multi-sheet workbooks (`pd.ExcelWriter`, `pd.ExcelFile`).
3. Perform initial exploratory data audits using `.info()`, `.describe()`, and `.shape`.
4. Inspect and convert data types (`dtypes`).
5. Drastically reduce memory footprints using category conversion and numeric downcasting.
6. **Advanced:** Construct, query, slice, and reshape Multi-Index (hierarchical) DataFrames with `pd.IndexSlice`.

In [1]:
import pandas as pd
import numpy as np
import io

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.6


### 1. The 1D Building Block: `pd.Series`

A `Series` is a 1-dimensional labeled array capable of holding any data type.
- Unlike a 1D NumPy array, a `Series` possesses an explicit **index** (row labels).
- Operations align automatically based on the index labels.

In [2]:
# Creating a Series from a Python list with custom index
temperatures = pd.Series([22.5, 24.0, 19.8, 25.2], index=['London', 'Paris', 'Berlin', 'Madrid'], name="Temperature_C")

print(temperatures)
print("\nIndex:  ", temperatures.index)
print("Values: ", temperatures.values)
print("Access by label ('Berlin'):", temperatures['Berlin'])
print("Vectorized conversion (Fahrenheit):\n", temperatures * 9/5 + 32)

London    22.5
Paris     24.0
Berlin    19.8
Madrid    25.2
Name: Temperature_C, dtype: float64

Index:   Index(['London', 'Paris', 'Berlin', 'Madrid'], dtype='str')
Values:  [22.5 24.  19.8 25.2]
Access by label ('Berlin'): 19.8
Vectorized conversion (Fahrenheit):
 London    72.50
Paris     75.20
Berlin    67.64
Madrid    77.36
Name: Temperature_C, dtype: float64


---
### 2. The 2D Tabular Engine: `pd.DataFrame`

A `DataFrame` represents a 2D tabular dataset where:
- Rows are indexed observations (samples $N$).
- Columns are named variables/features (dimensions $D$).
- Each column is internally an individual `pd.Series`.

In [3]:
# Creating a DataFrame from a dictionary of lists
data_dict = {
    'Customer_ID': [1001, 1002, 1003, 1004, 1005],
    'Age': [28, 45, 33, 54, 23],
    'Annual_Income': [55000.0, 85000.0, 62000.0, 110000.0, 42000.0],
    'Loyalty_Member': [True, True, False, True, False]
}

df_customers = pd.DataFrame(data_dict)
print("Customer DataFrame:\n", df_customers)
print("\nDataFrame Shape (Samples, Features):", df_customers.shape)

Customer DataFrame:
    Customer_ID  Age  Annual_Income  Loyalty_Member
0         1001   28        55000.0            True
1         1002   45        85000.0            True
2         1003   33        62000.0           False
3         1004   54       110000.0            True
4         1005   23        42000.0           False

DataFrame Shape (Samples, Features): (5, 4)


---
### 3. Tabular File I/O: Deep Dive into CSV and Excel (.xlsx) Files

In real-world data science and machine learning workflows, tabular datasets predominantly arrive in two formats:
1. **CSV (Comma-Separated Values)**: Lightweight, ubiquitous, plain-text tabular format.
2. **Excel (.xlsx)**: Ubiquitous in business and analytics environments, containing multiple sheets, formatted headers, and metadata.

Pandas provides high-performance, flexible readers and writers for both formats: `pd.read_csv`, `df.to_csv`, `pd.read_excel`, `df.to_excel`, and `pd.ExcelWriter`.

#### 3A. Working with CSV Files (`read_csv` & `to_csv`)

Key production configurations for `pd.read_csv()`:
- **`sep` / `delimiter`**: Defaults to `,`, but datasets often use tabs (`\t`), pipes (`|`), or semicolons (`;`).
- **`usecols`**: Loads only specified columns—drastically reduces memory consumption on wide tables.
- **`dtype`**: Enforces specific types at read-time, preventing expensive automatic type inference.
- **`na_values`**: Custom list of string tokens to interpret as `NaN` (e.g. `['?', 'NA', 'missing', '-']`).
- **`chunksize`**: Returns an iterator over batches, enabling out-of-core streaming of multi-gigabyte files.

In [4]:
import os

# Robust path detection for repository data_files directory
data_dir = "data_files" if os.path.exists("data_files") else "../data_files"
csv_path = os.path.join(data_dir, "telecom_churn.csv")

# 1. Standard CSV read directly from data_files/
df = pd.read_csv(csv_path)
print("Standard CSV read from data_files/ (first 3 rows):")
print(df.head(3))

# 2. Memory-optimized read: Selecting specific columns & enforcing dtypes
df_subset = pd.read_csv(
    csv_path,
    usecols=['customer_id', 'monthly_charges', 'churn'],
    dtype={'customer_id': 'string', 'monthly_charges': 'float32'}
)
print("\nMemory-Optimized Read (usecols + dtype):\n", df_subset.head(3))

# 3. Out-of-Core Batch Streaming via chunksize
print("\nProcessing Large CSV in Chunks (chunksize=3):")
total_revenue = 0.0
for chunk_idx, chunk in enumerate(pd.read_csv(csv_path, chunksize=3)):
    chunk_rev = chunk['monthly_charges'].sum()
    total_revenue += chunk_rev
    print(f"  Chunk {chunk_idx + 1}: {len(chunk)} rows | Chunk Revenue = ${chunk_rev:.2f}")

print(f"Total Streamed Revenue: ${total_revenue:.2f}")

Standard CSV read from data_files/ (first 3 rows):
  customer_id churn  monthly_charges  total_charges   contract_type  \
0        C-01    No            29.85          29.85  Month-to-month   
1        C-02   Yes            56.95        1889.50        One year   
2        C-03    No            53.85         108.15  Month-to-month   

   tenure_months  
0              1  
1             34  
2              2  

Memory-Optimized Read (usecols + dtype):
   customer_id churn  monthly_charges
0        C-01    No        29.850000
1        C-02   Yes        56.950001
2        C-03    No        53.849998

Processing Large CSV in Chunks (chunksize=3):
  Chunk 1: 3 rows | Chunk Revenue = $140.65
  Chunk 2: 3 rows | Chunk Revenue = $202.10
  Chunk 3: 3 rows | Chunk Revenue = $200.15
  Chunk 4: 1 rows | Chunk Revenue = $45.30
Total Streamed Revenue: $588.20


#### 3B. Working with Excel (.xlsx) Files (`read_excel`, `to_excel`, & `ExcelWriter`)

Unlike plain CSV files, Excel workbooks can contain **multiple sheets**, embedded formulas, and metadata banners:
- **`pd.ExcelWriter(..., engine='openpyxl')`**: Context manager to write multiple DataFrames to distinct sheets in a single `.xlsx` file.
- **`pd.ExcelFile`**: Reads the workbook structure and sheet names without loading full table data into RAM.
- **`pd.read_excel(..., sheet_name=...)`**: Reads a single sheet by name or index.
- **`pd.read_excel(..., sheet_name=None)`**: Loads **all sheets simultaneously** into a Python dictionary of DataFrames `{sheet_name: df}`.
- **`skiprows`**: Skips introductory title lines or metadata banners before table headers.

In [5]:
excel_path = os.path.join(data_dir, "business_metrics.xlsx")

# 1. Inspecting sheet names without loading full data using pd.ExcelFile
excel_meta = pd.ExcelFile(excel_path)
print("Available Sheet Names in business_metrics.xlsx:")
print(excel_meta.sheet_names)

# 2. Reading a specific sheet
df_plans_loaded = pd.read_excel(excel_path, sheet_name='Subscription_Plans')
print("\nLoaded 'Subscription_Plans' Sheet:\n", df_plans_loaded)

# 3. Loading all sheets into a dictionary of DataFrames
all_sheets = pd.read_excel(excel_path, sheet_name=None)
print("\nLoaded All Sheets Summary:")
for name, sheet_data in all_sheets.items():
    print(f"  Sheet '{name}': shape = {sheet_data.shape}")

# 4. Writing multiple DataFrames to an Excel Workbook using pd.ExcelWriter
export_path = os.path.join(data_dir, "exported_demo_report.xlsx")
with pd.ExcelWriter(export_path, engine='openpyxl') as writer:
    df.head(5).to_excel(writer, sheet_name='Top_Customers', index=False)
    df_plans_loaded.to_excel(writer, sheet_name='Plan_Rates', index=False)
print(f"\nExported customized multi-sheet report to: {export_path}")

# Clean up temporary export demonstration file
if os.path.exists(export_path):
    os.remove(export_path)

Available Sheet Names in business_metrics.xlsx:
['Customers', 'Subscription_Plans', 'Regional_Targets']

Loaded 'Subscription_Plans' Sheet:
     Plan_Tier  Monthly_Rate  Max_Users  Storage_GB
0       Basic         29.99          1          10
1    Standard         59.99          5          50
2     Premium         99.99         20         250
3  Enterprise        249.99        100        1000

Loaded All Sheets Summary:
  Sheet 'Customers': shape = (10, 6)
  Sheet 'Subscription_Plans': shape = (4, 4)
  Sheet 'Regional_Targets': shape = (4, 4)

Exported customized multi-sheet report to: ../data_files/exported_demo_report.xlsx


---
### 4. Exploratory Data Auditing: `head`, `info`, and `describe`

When receiving a new machine learning dataset, standard exploratory audits include:
1. `df.head(n)` / `df.tail(n)`: Inspect actual samples.
2. `df.info()`: Check memory usage, column names, and non-null counts.
3. `df.describe()`: Summary statistics for numeric and categorical features.

In [6]:
print("=== DataFrame Info ===")
df.info()

print("\n=== Numerical Feature Summary Statistics ===")
print(df.describe().round(2))

print("\n=== Categorical Summary Statistics ===")
print(df.describe(include=['str']))

=== DataFrame Info ===
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      10 non-null     str    
 1   churn            10 non-null     str    
 2   monthly_charges  10 non-null     float64
 3   total_charges    10 non-null     float64
 4   contract_type    10 non-null     str    
 5   tenure_months    10 non-null     int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 612.0 bytes

=== Numerical Feature Summary Statistics ===
       monthly_charges  total_charges  tenure_months
count            10.00          10.00          10.00
mean             58.82         894.38          14.90
std              24.46        1027.18          15.52
min              29.75          29.85           1.00
25%              43.05         180.56           2.25
50%              55.40         444.65           9.50
75%              69.42        1550.60  

---
### 5. Memory Optimization for Large-Scale Datasets

Default Pandas imports often load strings as `object` (pointer overhead) and numbers as 64-bit (`int64`, `float64`).
For datasets with millions of rows:
- Convert repetitive string columns with low cardinality to `category` dtype.
- Downcast numeric columns using `pd.to_numeric(..., downcast='integer'|'float')`.

In [7]:
initial_mem = df.memory_usage(deep=True).sum()

# 1. Convert categorical strings to 'category'
df['contract_type'] = df['contract_type'].astype('category')
df['churn'] = df['churn'].astype('category')

# 2. Downcast numeric columns
df['tenure_months'] = pd.to_numeric(df['tenure_months'], downcast='integer')
df['monthly_charges'] = pd.to_numeric(df['monthly_charges'], downcast='float')

optimized_mem = df.memory_usage(deep=True).sum()

print(f"Initial Memory:   {initial_mem} bytes")
print(f"Optimized Memory: {optimized_mem} bytes")
print(f"Memory reduction: {((initial_mem - optimized_mem) / initial_mem) * 100:.1f}%")
print("\nOptimized dtypes:\n", df.dtypes)

Initial Memory:   2015 bytes
Optimized Memory: 1092 bytes
Memory reduction: 45.8%

Optimized dtypes:
 customer_id             str
churn              category
monthly_charges     float32
total_charges       float64
contract_type      category
tenure_months          int8
dtype: object


---
### 6. Advanced Complex Usage: Hierarchical Indexing (MultiIndex) and Reshaping

In multi-dimensional or multi-tenant machine learning tasks (e.g., panel datasets, regional sales across quarterly cohorts, multi-sensor telemetry), data naturally exhibits hierarchical levels.

A `MultiIndex` (hierarchical index) allows you to represent higher-dimensional data within standard 2D DataFrames without losing tabular query semantics.

Key Operations:
1. `pd.MultiIndex.from_tuples` or `from_product` to establish levels.
2. `pd.IndexSlice` for multi-axis, multi-level coordinate slicing.
3. `unstack()` and `stack()` to pivot between hierarchical long and wide formats.
4. Level-specific aggregations across complex multi-index axes.

In [8]:
# Constructing a multi-level index representing regions, stores, and product categories
regions = ['North', 'South']
stores = ['Store_A', 'Store_B']
products = ['Electronics', 'Clothing']

# Create MultiIndex using Cartesian product
index = pd.MultiIndex.from_product([regions, stores], names=['Region', 'Store'])
columns = pd.MultiIndex.from_product([['Q1', 'Q2'], products], names=['Quarter', 'Product'])

# Generate synthetic revenue and sales volume
np.random.seed(42)
revenue_matrix = np.random.randint(50, 500, size=(4, 4)) * 1000

df_multi = pd.DataFrame(revenue_matrix, index=index, columns=columns)
print("Hierarchical MultiIndex DataFrame:")
display(df_multi) if 'display' in dir() else print(df_multi)

# 1. Coordinate slicing with pd.IndexSlice
idx = pd.IndexSlice
# Select all stores in 'North' region for 'Q1' Electronics
north_q1_elec = df_multi.loc[idx['North', :], idx['Q1', 'Electronics']]
print("\nNorth Region - Q1 Electronics Revenue:")
print(north_q1_elec)

# 2. Level-specific reduction aggregations
print("\nTotal Revenue Aggregated by Region (Level 0):")
print(df_multi.groupby(level='Region').sum())

# 3. Reshaping with unstack() and stack()
print("\nStacked by Product (Pivoting column level to row level):")
df_stacked = df_multi.stack(level='Product', future_stack=True)
print(df_stacked.head(6))

Hierarchical MultiIndex DataFrame:
Quarter                 Q1                   Q2         
Product        Electronics Clothing Electronics Clothing
Region Store                                            
North  Store_A      152000   485000      398000   320000
       Store_B      156000   121000      238000    70000
South  Store_A      152000   171000      264000   380000
       Store_B      137000   422000      149000   409000

North Region - Q1 Electronics Revenue:
Region  Store  
North   Store_A    152000
        Store_B    156000
Name: (Q1, Electronics), dtype: int64

Total Revenue Aggregated by Region (Level 0):
Quarter          Q1                   Q2         
Product Electronics Clothing Electronics Clothing
Region                                           
North        308000   606000      636000   390000
South        289000   593000      413000   789000

Stacked by Product (Pivoting column level to row level):
Quarter                         Q1      Q2
Region Store   Product

### Summary & Next Steps
In this notebook, you mastered:
- Properties of `pd.Series` and `pd.DataFrame`.
- Comprehensive file I/O for **CSV** and multi-sheet **Excel (`.xlsx`)** workbooks using `openpyxl`, `usecols`, `chunksize`, `pd.ExcelFile`, and `pd.ExcelWriter`.
- Exploratory schema auditing with `info()`, `describe()`, and `dtypes`.
- Categorical and numeric memory downcasting techniques.
- Hierarchical MultiIndex creation, coordinate slicing with `pd.IndexSlice`, and unstack/stack reshaping.

**Next Notebook:** `02_indexing_filtering_and_assignment.ipynb` — Master `.loc` vs `.iloc`, compound boolean filtering, `.query()`, method chaining, and vectorizing conditional logic.